In [4]:
#!pip install transformers opencv-python pandas

In [5]:
from getpass import getpass

API_KEY = getpass('Key: ')

In [8]:
from pathlib import Path
import base64
import cv2
import pandas as pd
import requests


import html
import re

OUTPUT_FILE = Path("submission.html")
DATASET_FOLDER = Path("../ImageDataset")

MODEL_NAME = "openai/gpt-5.5"

PROMPT = (
    "Return the answer using exactly two XML tags: "
    "<heading> and <description>. "
    "Inside <heading>, write a short and specific title of 2–6 words that clearly identifies what is visible in the image. "
    "Inside <description>, identify the main subject and describe it in one concise, factual paragraph of 2–3 sentences. "
    "Include distinctive colors, materials, shapes, context, and any readable text that would help a visual search engine find similar objects or scenes. "
    "Avoid decorative language, unsupported assumptions, and phrases such as 'The image shows.' "
    "Express uncertain identifications cautiously using words such as 'likely,' 'possibly,' or 'appears to be.' "
    "Use this exact output format: "
    "<heading>Specific image title</heading>"
    "<description>Concise factual description.</description>"
)

def describe_image(image_path):
    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    height, width = image.shape[:2]
    max_dimension = 1600

    scale = min(max_dimension / width, max_dimension / height, 1.0)

    if scale < 1.0:
        new_width = round(width * scale)
        new_height = round(height * scale)

        image = cv2.resize(
            image,
            (new_width, new_height),
            interpolation=cv2.INTER_AREA,
        )

    success, image_buffer = cv2.imencode(
        ".jpg",
        image,
        [cv2.IMWRITE_JPEG_QUALITY, 90],
    )

    image_b64 = base64.b64encode(image_buffer).decode("utf-8")

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": PROMPT,
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": (
                                "data:image/jpeg;base64,"
                                + image_b64
                            )
                        },
                    },
                ],
            }
        ],
    }

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        json=payload,
        timeout=120,
    )

    response.raise_for_status()

    return response.json()["choices"][0]["message"]["content"]


accepted_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
}

image_paths = sorted(
    path
    for path in DATASET_FOLDER.iterdir()
    if path.suffix.lower() in accepted_extensions
)

print(f"Found {len(image_paths)} images.")

results = []

for index, image_path in enumerate(image_paths, start=1):
    print(f"[{index}/{len(image_paths)}] Processing: {image_path.name}")

    try:
        response = describe_image(image_path)

        heading_match = re.search(
            r"<heading>(.*?)</heading>",
            response,
            re.DOTALL | re.IGNORECASE,
        )

        description_match = re.search(
            r"<description>(.*?)</description>",
            response,
            re.DOTALL | re.IGNORECASE,
        )

        heading = (
            heading_match.group(1).strip()
            if heading_match
            else "Missing heading"
        )

        description = (
            description_match.group(1).strip()
            if description_match
            else response
        )

    except Exception as error:
        heading = "Error"
        description = str(error)

    results.append(
        {
            "image_path": image_path.as_posix(),
            "heading": heading,
            "description": description,
        }
    )


html_cards = ""

for result in results:
    html_cards += f"""
    <article class="card">
        <img
            src="{html.escape(result['image_path'])}"
            alt="{html.escape(result['heading'])}"
        >
        <div class="content">
            <h2>{html.escape(result['heading'])}</h2>
            <p>{html.escape(result['description'])}</p>
            <small>{html.escape(result['image_path'])}</small>
        </div>
    </article>
    """


html_page = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>OpenLens Results</title>

    <style>
        body {{
            margin: 0;
            padding: 40px;
            font-family: Arial, sans-serif;
            background: #0b1220;
            color: white;
        }}

        h1 {{
            text-align: center;
            margin-bottom: 40px;
        }}

        .results {{
            max-width: 900px;
            margin: auto;
            display: grid;
            gap: 24px;
        }}

        .card {{
            display: flex;
            gap: 24px;
            padding: 20px;
            background: #152238;
            border-radius: 14px;
        }}

        .card img {{
            width: 230px;
            height: 180px;
            object-fit: cover;
            border-radius: 10px;
        }}

        .card h2 {{
            margin-top: 0;
            color: #65d6c3;
        }}

        .card p {{
            line-height: 1.6;
            color: #dce5f2;
        }}

        .card small {{
            color: #91a0b5;
        }}

        @media (max-width: 650px) {{
            .card {{
                flex-direction: column;
            }}

            .card img {{
                width: 100%;
            }}
        }}
    </style>
</head>

<body>
    <h1>OpenLens Image Descriptions</h1>

    <main class="results">
        {html_cards}
    </main>
</body>
</html>
"""


OUTPUT_FILE.write_text(html_page, encoding="utf-8")

print("\nEvaluation completed.")
print(f"Results were saved to: {OUTPUT_FILE.resolve()}")

Found 16 images.
[1/16] Processing: Apples.jpg
[2/16] Processing: Bicycles_parked.jpg
[3/16] Processing: Blurry_city.jpg
[4/16] Processing: Chair.jpg
[5/16] Processing: Chair2.jpg
[6/16] Processing: Coffee_in_a_mug.jpg
[7/16] Processing: Cute_seal.jpg
[8/16] Processing: Dog_Couch.jpg
[9/16] Processing: Kitchen_interior_design.jpg
[10/16] Processing: Laptop_on_a_desk.jpg
[11/16] Processing: Pedestrians_crossing_the_a_busy_street.jpg
[12/16] Processing: Picture_of_computer_desk.jpg
[13/16] Processing: Riding_a_bike_in_the_park.jpg
[14/16] Processing: Seal.jpg
[15/16] Processing: Traffic_Sign_post.jpg
[16/16] Processing: White_mug.jpg

Evaluation completed.
Results were saved to: C:\Users\ROG\openlens\Model\BestPrompt\submission.html
